In [1]:
pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'c:\Users\vaish\OneDrive\Desktop\Dataset\PJME_hourly.csv',
                 parse_dates=['Datetime'], index_col='Datetime')

df_all = pd.read_csv(r'c:\Users\vaish\OneDrive\Desktop\Dataset\pjm_hourly_est.csv',
                     parse_dates=['Datetime'], index_col='Datetime')

In [4]:
print(df.shape)
print(df.info())
print(df.describe())
print(f"Date range: {df.index.min()} → {df.index.max()}")
print(f"Nulls:\n{df.isnull().sum()}")
print(f"Duplicated timestamps: {df.index.duplicated().sum()}")

(145366, 1)
<class 'pandas.DataFrame'>
DatetimeIndex: 145366 entries, 2002-12-31 01:00:00 to 2018-01-02 00:00:00
Data columns (total 1 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   PJME_MW  145366 non-null  float64
dtypes: float64(1)
memory usage: 2.2 MB
None
             PJME_MW
count  145366.000000
mean    32080.222831
std      6464.012166
min     14544.000000
25%     27573.000000
50%     31421.000000
75%     35650.000000
max     62009.000000
Date range: 2002-01-01 01:00:00 → 2018-08-03 00:00:00
Nulls:
PJME_MW    0
dtype: int64
Duplicated timestamps: 4


In [6]:
df = df[~df.index.duplicated(keep='first')]

In [7]:
df = df.sort_index()

full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
print(f"Missing hours: {len(full_idx) - len(df)}")

df = df.reindex(full_idx)
df.index.name = 'Datetime'

Missing hours: 30


In [8]:
#Interpolated 
df = df.interpolate(method='time')


In [9]:
col = df.columns[0]  # e.g., 'PJME_MW'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = (df[col] < lower) | (df[col] > upper)
print(f"Outliers found: {outliers.sum()} ({outliers.mean()*100:.2f}%)")

# Capped Outliers
df[col] = df[col].clip(lower=lower, upper=upper)



Outliers found: 3460 (2.38%)


In [10]:
df['hour']        = df.index.hour
df['dayofweek']   = df.index.dayofweek      # 0=Mon, 6=Sun
df['month']       = df.index.month
df['year']        = df.index.year
df['dayofyear']   = df.index.dayofyear
df['quarter']     = df.index.quarter
df['is_weekend']  = df['dayofweek'].isin([5, 6]).astype(int)

df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [11]:
col = df.columns[0]
df['lag_1h']  = df[col].shift(1)       # 1 hour ago
df['lag_24h'] = df[col].shift(24)      # same hour yesterday
df['lag_168h'] = df[col].shift(168)    # same hour last week

df['rolling_24h_mean'] = df[col].rolling(24).mean()
df['rolling_24h_std']  = df[col].rolling(24).std()

df = df.dropna()

In [12]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler = MinMaxScaler()
df['MW_scaled'] = scaler.fit_transform(df[[col]])


In [13]:
split_date = '2017-01-01'
train = df.loc[:split_date]
test  = df.loc[split_date:]

print(f"Train: {train.shape[0]} rows ({train.index.min()} → {train.index.max()})")
print(f"Test:  {test.shape[0]} rows ({test.index.min()} → {test.index.max()})")

Train: 131351 rows (2002-01-08 01:00:00 → 2017-01-01 23:00:00)
Test:  13897 rows (2017-01-01 00:00:00 → 2018-08-03 00:00:00)


In [14]:
df.to_csv(r'C:\Users\vaish\OneDrive\Desktop\Dataset\Preprocessed\PJME_preprocessed.csv')
